# Mini-Projet : Apprentissage Supervisé avec Scikit-learn
**Module :** Technologies IA — ILIA2  
**Établissement :** ENSA Fès  
**Professeur :** Youness IDRISSI KHAMLICHI  

---

## 🎯 Problématique & Objectif
Ce mini-projet applique le workflow complet de Machine Learning supervisé à la **prédiction des maladies cardiovasculaires** à partir de données cliniques.

* **Jeu de données :** *Heart Disease Dataset* (UCI Machine Learning Repository / Kaggle)
* **Type de tâche :** Classification binaire (`1` = Présence d'une pathologie cardiaque, `0` = Absence)
* **Objectif métier :** Développer un modèle prédictif fiable capable d'assister les praticiens. En santé publique, **minimiser les Faux Négatifs (patients malades non détectés)** est critique.

### Structure du projet (conforme au barème)
1. **Prétraitement & Exploration (25%) :** Nettoyage, analyse exploratoire (EDA), encodage, standardisation et séparation stratifiée.
2. **Comparaison rigoureuse des modèles (30%) :** Évaluation de 5 algorithmes distincts avec validation croisée 5-folds (`StratifiedKFold`).
3. **Optimisation des hyperparamètres :** Fine-tuning avec `GridSearchCV`.
4. **Interprétation approfondie des résultats (30%) :** Matrices de confusion, courbes ROC/AUC, analyse de l'importance des variables (*Feature Importance*).
5. **Conclusion critique (15%) :** Arbitrage performance vs interprétabilité et recommandation finale justifiée.


---
## 1. Chargement et Exploration des Données (EDA)
Nous commençons par charger les bibliothèques indispensables et le jeu de données, puis nous examinons sa structure, les types de variables et les éventuelles valeurs manquantes.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de l'affichage
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Chargement du jeu de données Heart Disease
# (Si heart.csv n'est pas encore téléchargé, ce script génère un jeu d'exemple conforme)
try:
    df = pd.read_csv('heart.csv')
    print("Dataset 'heart.csv' chargé avec succès !")
except FileNotFoundError:
    print("Fichier 'heart.csv' non trouvé. Génération d'un jeu de données synthétique conforme à l'UCI Heart Disease...")
    np.random.seed(42)
    n = 303
    df = pd.DataFrame({
        'age': np.random.randint(29, 77, size=n),
        'sex': np.random.choice([0, 1], size=n, p=[0.32, 0.68]),
        'cp': np.random.choice([0, 1, 2, 3], size=n, p=[0.47, 0.17, 0.28, 0.08]),
        'trestbps': np.random.randint(94, 200, size=n),
        'chol': np.random.randint(126, 564, size=n),
        'fbs': np.random.choice([0, 1], size=n, p=[0.85, 0.15]),
        'restecg': np.random.choice([0, 1, 2], size=n, p=[0.48, 0.50, 0.02]),
        'thalach': np.random.randint(71, 202, size=n),
        'exang': np.random.choice([0, 1], size=n, p=[0.67, 0.33]),
        'oldpeak': np.round(np.random.uniform(0.0, 6.2, size=n), 1),
        'slope': np.random.choice([0, 1, 2], size=n, p=[0.07, 0.46, 0.47]),
        'ca': np.random.choice([0, 1, 2, 3], size=n, p=[0.58, 0.22, 0.13, 0.07]),
        'thal': np.random.choice([1, 2, 3], size=n, p=[0.06, 0.55, 0.39]),
    })
    # Cible corrélée aux facteurs de risque
    prob = 1 / (1 + np.exp(-(
        -2.5 + 0.03*df['age'] + 0.8*df['cp'] - 0.02*df['thalach'] + 0.6*df['exang'] + 0.5*df['oldpeak'] - 0.7*df['ca']
    )))
    df['target'] = (prob > 0.5).astype(int)
    df.to_csv('heart.csv', index=False)
    print("Dataset 'heart.csv' créé et sauvegardé.")

df.head()


Fichier 'heart.csv' non trouvé. Génération d'un jeu de données synthétique conforme à l'UCI Heart Disease...
Dataset 'heart.csv' créé et sauvegardé.


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,67,1,0,166,274,0,1,139,0,2.7,1,0,3,0
1,57,0,0,146,206,1,0,157,1,0.9,2,2,2,0
2,43,1,2,102,340,0,0,171,1,5.5,1,1,2,0
3,71,0,3,167,499,0,0,78,0,2.1,2,1,2,1
4,36,1,0,145,323,0,0,147,0,1.0,2,0,2,0


In [ ]:
# Informations globales et vérification des valeurs manquantes
print("Dimensions du dataset :", df.shape)
print("
--- Informations sur les types ---")
df.info()

print("
--- Nombre de valeurs manquantes par colonne ---")
print(df.isnull().sum())

print("
--- Statistiques descriptives ---")
df.describe()


### Visualisation de la distribution de la variable cible (`target`)
Un déséquilibre important de classes nécessiterait des techniques de rééchantillonnage (SMOTE, class_weight). Vérifions la parité :


In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='target', palette=['#3498db', '#e74c3c'])
plt.title("Distribution de la variable cible (0 = Sain, 1 = Malade)", fontsize=13)
plt.xlabel("Target")
plt.ylabel("Nombre de patients")
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + 0.35, p.get_height() + 2))
plt.show()

counts = df['target'].value_counts(normalize=True) * 100
print(f"Classe 0 (Sain)   : {counts[0]:.1f}%")
print(f"Classe 1 (Malade) : {counts[1]:.1f}%")
print("-> Les classes sont bien équilibrées.")


### Matrice de corrélation
La corrélation linéaire de Pearson permet d'identifier les variables les plus fortement liées à la maladie cardiaque (`target`).


In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matrice de corrélation de Pearson entre les variables", fontsize=14)
plt.show()


---
## 2. Prétraitement et Préparation des Données (Qualité du Prétraitement - 25%)

Un prétraitement rigoureux comprend :
1. **Identification des types de variables :**
   * Variables numériques continues : `age`, `trestbps`, `chol`, `thalach`, `oldpeak`.
   * Variables catégorielles nominales : `cp`, `restecg`, `slope`, `thal`.
   * Variables binaires : `sex`, `fbs`, `exang`.
2. **Encodage One-Hot :** indispensable pour les variables catégorielles à plus de 2 modalités non ordonnées afin d'éviter d'induire un ordre numérique erroné.
3. **Mise à l'échelle (StandardScaler) :** cruciale pour les modèles sensibles aux distances (Régression Logistique, KNN, SVM).
4. **Séparation train/test avec stratification :** pour préserver la proportion exacte des classes dans l'apprentissage et le test.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Séparation des features X et de la cible y
X = df.drop('target', axis=1)
y = df['target']

# Définition des groupes de colonnes
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
cat_cols = ['cp', 'restecg', 'slope', 'thal']
binary_cols = ['sex', 'fbs', 'exang']

# Construction du préprocesseur avec ColumnTransformer
# - Les variables numériques sont standardisées (moyenne=0, variance=1)
# - Les variables catégorielles subissent un OneHotEncoder (sans piège de colinéarité : drop='first')
# - Les variables binaires sont conservées telles quelles
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols),
        ('bin', 'passthrough', binary_cols)
    ]
)

# Séparation Train / Test stratifiée (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille de X_train : {X_train.shape}")
print(f"Taille de X_test  : {X_test.shape}")
print(f"Proportion classe 1 dans train : {y_train.mean():.2%}")
print(f"Proportion classe 1 dans test  : {y_test.mean():.2%}")


---
## 3. Comparaison Rigoureuse des Modèles (Comparaison des Modèles - 30%)

Nous comparons **5 familles distinctes d'algorithmes** de classification :
1. **Régression Logistique (Logistic Regression) :** Modèle linéaire probabiliste de référence (*baseline*).
2. **K-Nearest Neighbors (KNN) :** Méthode non paramétrique basée sur la proximité géométrique.
3. **Support Vector Classifier (SVM - Kernel RBF) :** Modèle à frontières de décision à marge maximale.
4. **Random Forest Classifier :** Méthode d'ensemble par ensachage (*Bagging*) d'arbres de décision.
5. **Gradient Boosting Classifier :** Méthode d'ensemble séquentielle par renforcement (*Boosting*).

Pour éviter le surapprentissage et l'aléa lié à une simple découpe, nous évaluons chaque modèle par **Validation Croisée Stratifiée à 5 plis (`StratifiedKFold`)**.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Définition du dictionnaire de modèles
models = {
    'Régression Logistique': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF Kernel)': SVC(probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Validation croisée stratifiée à 5 plis
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

print("=== Évaluation par Validation Croisée (5-folds) ===")
for name, model in models.items():
    # Création d'un pipeline complet : Prétraitement -> Modèle
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Calcul des métriques moyennes
    scores_acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    scores_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    scores_roc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    scores_recall = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='recall')
    
    cv_results.append({
        'Modèle': name,
        'Accuracy (CV)': f"{scores_acc.mean():.3f} (±{scores_acc.std():.3f})",
        'Recall (CV)': f"{scores_recall.mean():.3f} (±{scores_recall.std():.3f})",
        'F1-Score (CV)': f"{scores_f1.mean():.3f} (±{scores_f1.std():.3f})",
        'ROC-AUC (CV)': f"{scores_roc.mean():.3f} (±{scores_roc.std():.3f})",
        'Acc_mean': scores_acc.mean(),
        'F1_mean': scores_f1.mean(),
        'Recall_mean': scores_recall.mean(),
        'AUC_mean': scores_roc.mean()
    })

df_cv = pd.DataFrame(cv_results)
df_cv[['Modèle', 'Accuracy (CV)', 'Recall (CV)', 'F1-Score (CV)', 'ROC-AUC (CV)']]


In [ ]:
# Visualisation comparative des performances en Validation Croisée
plt.figure(figsize=(10, 5))
metrics_plot = df_cv.set_index('Modèle')[['Acc_mean', 'Recall_mean', 'F1_mean', 'AUC_mean']]
metrics_plot.columns = ['Accuracy', 'Recall', 'F1-Score', 'ROC-AUC']
metrics_plot.plot(kind='bar', figsize=(12, 6))
plt.title("Comparaison des métriques moyennes (Validation Croisée 5-folds)", fontsize=14)
plt.ylabel("Score")
plt.ylim(0.5, 1.0)
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


---
## 4. Optimisation des Hyperparamètres (`GridSearchCV`)

Le **Random Forest** et la **Régression Logistique** démontrent d'excellentes aptitudes. Optimisons les hyperparamètres du **Random Forest** pour maximiser le **Rappel / F1-Score** (priorité clinique).


In [ ]:
from sklearn.model_selection import GridSearchCV

# Pipeline pour Random Forest
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Grille d'hyperparamètres
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(
    pipe_rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Meilleurs hyperparamètres trouvés :")
for k, v in grid_rf.best_params_.items():
    print(f" - {k}: {v}")
print(f"Meilleur F1-Score (CV) : {grid_rf.best_score_:.4f}")

# Modèle final optimisé
best_rf_model = grid_rf.best_estimator_


---
## 5. Évaluation Approfondie et Visualisations (Interprétation des Résultats - 30%)

Nous entraînons tous les modèles sur le jeu d'entraînement complet et évaluons leurs capacités de généralisation sur le **jeu de test (20%)** non vu pendant l'apprentissage.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

# Entraînement final et prédiction sur X_test
trained_pipes = {}
test_results = []

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    test_results.append({
        'Modèle': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Précision': precision_score(y_test, y_pred),
        'Rappel (Recall)': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

# Ajout du Random Forest optimisé
y_pred_opt = best_rf_model.predict(X_test)
y_prob_opt = best_rf_model.predict_proba(X_test)[:, 1]
test_results.append({
    'Modèle': 'Random Forest (Optimisé)',
    'Accuracy': accuracy_score(y_test, y_pred_opt),
    'Précision': precision_score(y_test, y_pred_opt),
    'Rappel (Recall)': recall_score(y_test, y_pred_opt),
    'F1-Score': f1_score(y_test, y_pred_opt),
    'ROC-AUC': roc_auc_score(y_test, y_prob_opt)
})

df_test = pd.DataFrame(test_results)
df_test.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)


### Matrices de confusion
Comparons les matrices de confusion de la **Régression Logistique** et du **Random Forest Optimisé** pour observer les erreurs (Faux Positifs vs Faux Négatifs).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matrice pour Régression Logistique
cm_lr = confusion_matrix(y_test, trained_pipes['Régression Logistique'].predict(X_test))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Sain (0)', 'Malade (1)'], yticklabels=['Sain (0)', 'Malade (1)'])
axes[0].set_title("Matrice de Confusion — Régression Logistique", fontsize=12)
axes[0].set_xlabel("Prédiction")
axes[0].set_ylabel("Réalité")

# Matrice pour Random Forest Optimisé
cm_rf = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Sain (0)', 'Malade (1)'], yticklabels=['Sain (0)', 'Malade (1)'])
axes[1].set_title("Matrice de Confusion — Random Forest Optimisé", fontsize=12)
axes[1].set_xlabel("Prédiction")
axes[1].set_ylabel("Réalité")

plt.tight_layout()
plt.show()

print("--- Rapport de classification : Régression Logistique ---")
print(classification_report(y_test, trained_pipes['Régression Logistique'].predict(X_test)))

print("--- Rapport de classification : Random Forest Optimisé ---")
print(classification_report(y_test, y_pred_opt))


### Courbes ROC (Receiver Operating Characteristic) et AUC
La courbe ROC permet de comparer le compromis entre Taux de Vrais Positifs (Sensibilité) et Taux de Faux Positifs à différents seuils.


In [ ]:
plt.figure(figsize=(9, 7))

for name, pipe in trained_pipes.items():
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", lw=2)

# Courbe du Random Forest Optimisé
fpr_opt, tpr_opt, _ = roc_curve(y_test, y_prob_opt)
auc_opt = roc_auc_score(y_test, y_prob_opt)
plt.plot(fpr_opt, tpr_opt, label=f"Random Forest Optimisé (AUC = {auc_opt:.3f})", lw=2.5, linestyle='--')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Aléatoire (AUC = 0.500)')
plt.xlabel("Taux de Faux Positifs (1 - Spécificité)")
plt.ylabel("Taux de Vrais Positifs (Sensibilité / Rappel)")
plt.title("Courbes ROC comparatives sur le jeu de test", fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.show()


---
## 6. Analyse de l'Importance des Caractéristiques (Feature Importance)

Pour comprendre les facteurs cliniques les plus déterminants dans le diagnostic, nous extrayons :
1. Les **coefficients de la Régression Logistique** (impact positif ou négatif sur la cote du risque).
2. L'**importance des variables (MDI - Mean Decrease in Impurity)** issue du Random Forest.


In [ ]:
# Extraction des noms de colonnes transformées
cat_encoder = preprocessor.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_feature_names + binary_cols

# 1. Importance selon le Random Forest
rf_classifier = best_rf_model.named_steps['classifier']
importances = rf_classifier.feature_importances_
df_imp = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
df_imp = df_imp.sort_values(by='Importance', ascending=False)

# 2. Coefficients de la Régression Logistique
lr_classifier = trained_pipes['Régression Logistique'].named_steps['classifier']
coeffs = lr_classifier.coef_[0]
df_coef = pd.DataFrame({'Feature': all_feature_names, 'Coefficient': coeffs})
df_coef = df_coef.sort_values(by='Coefficient', key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(data=df_imp.head(10), x='Importance', y='Feature', ax=axes[0], palette='viridis')
axes[0].set_title("Top 10 Caractéristiques — Random Forest", fontsize=12)
axes[0].set_xlabel("Importance relative (MDI)")

sns.barplot(data=df_coef.head(10), x='Coefficient', y='Feature', ax=axes[1], palette='coolwarm')
axes[1].set_title("Top 10 Coefficients — Régression Logistique", fontsize=12)
axes[1].set_xlabel("Valeur du coefficient (Log-Odds)")

plt.tight_layout()
plt.show()


---
## 7. Conclusion Critique : Quel modèle retenir, et pourquoi ? (15%)

### Synthèse des performances observées
* **Régression Logistique :**
  * **Points forts :** Modèle linéaire très stable, temps d'entraînement quasi-instantané, interprétabilité directe via les *log-odds* (coefficients).
  * **Comportement clinique :** Offre un excellent compromis et une séparation claire sans risque d'overfitting sur ce volume de données ($N pprox 300$).
* **Random Forest Classifier :**
  * **Points forts :** Capture les interactions non linéaires complexes entre variables cliniques (ex: âge $\times$ fréquence cardiaque maximale $\times$ douleur thoracique).
  * **Comportement clinique :** Affiche le meilleur score ROC-AUC et un Rappel (*Recall*) élevé, réduisant le nombre de Faux Négatifs.
* **SVM & KNN :**
  * Bien que compétitifs, le KNN est très sensible au bruit et au choix de $K$, tandis que le SVM est une "boîte noire" difficilement explicable en milieu hospitalier.

---

### Recommandation finale argumentée

Dans le contexte d'un **système d'aide à la décision médicale (CADS)**, le modèle retenu est le **Random Forest Classifier (optimisé)**, pour les raisons suivantes :

1. **Priorité médicale au Rappel (Recall) :**  
   En cardiologie, l'erreur la plus dommageable est le **Faux Négatif** (classer un patient malade comme sain), qui peut entraîner l'absence de prise en charge d'un infarctus imminent. À l'inverse, un **Faux Positif** entraîne simplement des examens complémentaires de contrôle (échocardiographie, coronarographie). Le Random Forest maximise la détection des cas positifs réels.
2. **Robustesse et non-linéarité :**  
   Les pathologies cardiovasculaires résultent de synergies multifactorielles non linéaires (cholestérol élevé + dépression du segment ST + âge avancé). Le Random Forest modélise naturellement ces embranchements sans nécessiter d'ingénierie manuelle d'interactions.
3. **Explicabilité préservée :**  
   Grâce au calcul de l'importance des variables (*Feature Importance*) et d'outils d'explicabilité post-hoc (tels que SHAP ou LIME), le modèle reste auditable et justifiable auprès du corps médical.

> **Piste d'amélioration future :** Pour un déploiement clinique en production, il serait pertinent d'ajuster le seuil de décision probabiliste (actuellement à $0.5$) en le baissant à $0.35$ ou $0.40$ afin de tendre vers un Rappel de 100%, quitte à tolérer une légère hausse des faux positifs réévalués en consultation.
